# Retrain on a Colab T4, with Drive checkpoints

GPU runtime. This notebook clones `main`, installs with `pip install -e '.[dev]'` (no virtualenv; Colab's `ensurepip` fails), mounts Drive, uploads a labels-and-manifests zip, and resumes training.

After every epoch `checkpoints/last.pt` is copied to `/content/drive/MyDrive/afroxlmr/last.pt`. Re-run the training cell after a disconnect and it resumes from that file.

The published inference checkpoint is `afroxlmr/model.pt` in https://drive.google.com/drive/folders/1Ql5b645nf9j_UhWygj7kYqH0jaWZsDl9?usp=sharing . SHA-256 `778246a31841ac554fbf574cfc42cc9579981b67fb85a8eb1814dbb689ea9eb6`. That file is the export, not the resume checkpoint.

Do not commit the upload. This notebook stores no outputs.


In [ ]:
%cd /content
!if [ ! -d /content/capstone/.git ]; then git clone --branch main --single-branch https://github.com/ChristopherKiokoStrathmore/-DSA8401---ML-CAPSTONE-PROJECT-.git /content/capstone; fi
%cd /content/capstone
!git pull --ff-only
!pip install -q -e '.[dev]'


Mount Drive. The resume file lives at `MyDrive/afroxlmr/last.pt`. Create the directory if this is the first run.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
drive_dir = Path('/content/drive/MyDrive/afroxlmr')
drive_dir.mkdir(parents=True, exist_ok=True)
print('drive', drive_dir)
print('last.pt', (drive_dir / 'last.pt').is_file())


Upload one zip. It must contain `labels/` and `manifests/`.

`labels/` uses the canonical names in `docs/label_rubric.md`: `issue.csv`, `sentiment_hand.csv`, `sentiment_platform.csv`, `sentiment_test.csv`, `urgency_train.csv`, `urgency_test.csv`, `gemini_relabeled.csv`.

`manifests/` is the JSON from `make labels` (`issue.json`, `sentiment.json`, `urgency.json`, `registry.json`).


In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

destination = Path('/content/data')
destination.mkdir(parents=True, exist_ok=True)
print('Upload the labels + manifests zip.')
uploaded = files.upload()
for name, payload in uploaded.items():
    path = destination / name
    path.write_bytes(payload)
    if path.suffix == '.zip':
        with zipfile.ZipFile(path) as archive:
            archive.extractall(destination)
        print('extracted', path)
print('labels', list((destination / 'labels').glob('*.csv')))
print('manifests', list((destination / 'manifests').glob('*.json')))


If Drive already has `last.pt`, copy it into the checkpoint directory before `--resume`. The first run has no file and starts at epoch 1.


In [ ]:
import shutil
from pathlib import Path

checkpoint_dir = Path('/content/capstone/artifacts/afroxlmr/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)
saved = Path('/content/drive/MyDrive/afroxlmr/last.pt')
if saved.is_file() and saved.stat().st_size > 0:
    shutil.copy2(saved, checkpoint_dir / 'last.pt')
    print('resume from', saved, saved.stat().st_size)
else:
    print('no Drive checkpoint; training starts at epoch 1')


In [ ]:
%cd /content/capstone
!python -m capstone.models.train \
  --config configs/afroxlmr.yaml \
  --labels /content/data/labels \
  --manifests /content/data/manifests \
  --out artifacts/afroxlmr \
  --kind data \
  --resume \
  --mirror /content/drive/MyDrive/afroxlmr


After training, print the SHA-256 of the exported `model.pt` and copy it next to the resume file. Compare the digest with `778246a31841ac554fbf574cfc42cc9579981b67fb85a8eb1814dbb689ea9eb6`.


In [ ]:
import hashlib
import shutil
from pathlib import Path

weights = Path('/content/capstone/artifacts/afroxlmr/model.pt')
digest = hashlib.sha256(weights.read_bytes()).hexdigest()
print(digest)
print(Path('/content/capstone/artifacts/afroxlmr/sha256.txt').read_text())
shutil.copy2(weights, '/content/drive/MyDrive/afroxlmr/model.pt')
print('copied model.pt')
